# Transfer Learning: ResNet50 Fine-Tuning on CIFAR-10

Builds on the baseline CNN (`01-cifar10-training.ipynb`). This notebook loads a
**pretrained ResNet50** (ImageNet weights), freezes its base, trains a new classifier
head on CIFAR-10, then **fine-tunes** the top layers. Expected test accuracy: **90–95%**.

**How to use:** Google Colab → Upload → Runtime → **Run all** (GPU).

> Note: ResNet50 downloads ~90 MB of ImageNet weights on first run.

In [ ]:
# Install extra packages (TensorFlow is pre-installed in Colab)
%pip install -q numpy matplotlib scikit-learn

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, applications
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

In [ ]:
# ---------- Configuration ----------
BATCH_SIZE  = 64
EPOCHS_HEAD = 15
EPOCHS_FT   = 10
LR_HEAD     = 0.001
LR_FINE     = 0.00001      # very small LR for fine-tuning
IMG_SIZE    = 224          # ResNet50 input size
NUM_CLASSES = 10
PATIENCE    = 5

CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]

MODELS_DIR  = "/content/models"
RESULTS_DIR = "/content/results"
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

## 1) Data Pipeline

Load CIFAR-10, one-hot encode, split 85/15 (train/validation), then build **tf.data**
pipelines that resize images to 224x224 and scale to [-1, 1] (ResNet50 expects this).

In [ ]:
def load_cifar10():
    (x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
    return x_train, y_train, x_test, y_test

def one_hot(y, num_classes=NUM_CLASSES):
    return keras.utils.to_categorical(y, num_classes)

def split_train_val(x, y, val_ratio=0.15, seed=SEED):
    indices = np.arange(len(x))
    rng = np.random.RandomState(seed)
    rng.shuffle(indices)
    n_val = int(len(indices) * val_ratio)
    val_idx, train_idx = indices[:n_val], indices[n_val:]
    return x[train_idx], y[train_idx], x[val_idx], y[val_idx]

x_train, y_train, x_test, y_test = load_cifar10()
x_train, y_train, x_val, y_val = split_train_val(x_train, y_train)

# Normalize here; [-1, 1] scaling happens in the pipeline map below
x_train = x_train.astype("float32") / 255.0
x_val   = x_val.astype("float32") / 255.0
x_test  = x_test.astype("float32") / 255.0
y_train = one_hot(y_train)
y_val   = one_hot(y_val)
y_test  = one_hot(y_test)

print("train:", x_train.shape, "| val:", x_val.shape, "| test:", x_test.shape)

In [ ]:
def make_tf_dataset(x, y, batch_size=BATCH_SIZE, augment=False, img_size=IMG_SIZE):
    ds = tf.data.Dataset.from_tensor_slices((x, y))

    def preprocess(im, lbl):
        im = tf.image.resize(im, (img_size, img_size))  # 32x32 -> 224x224
        im = im * 2.0 - 1.0                             # [0,1] -> [-1,1]
        if augment:
            im = tf.image.random_flip_left_right(im)
        return im, lbl

    ds = ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.shuffle(10000)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_tf_dataset(x_train, y_train, augment=True)
val_ds   = make_tf_dataset(x_val, y_val)
test_ds  = make_tf_dataset(x_test, y_test)

for batch in train_ds.take(1):
    print("batch images:", batch[0].shape, "labels:", batch[1].shape)

## 2) Build the Model: Frozen ResNet50 Base + New Head

In [ ]:
base = applications.ResNet50(
    weights="imagenet",
    include_top=False,          # drop ImageNet classifier
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    pooling="avg",              # global average pooling -> 2048-d vector
)
base.trainable = False          # phase 1: freeze pretrained weights

model = keras.Sequential([
    base,
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation="softmax"),
], name="resnet50_cifar10")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR_HEAD),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

## 3) Phase 1 — Train the New Head (feature extraction)

Only the new Dense layers are trained; the base stays frozen.

In [ ]:
cb = [
    callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True, verbose=1),
    callbacks.ModelCheckpoint(os.path.join(MODELS_DIR, "resnet50_head_best.keras"),
                              monitor="val_accuracy", save_best_only=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1),
]

history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    callbacks=cb,
    verbose=1,
)

In [ ]:
def plot_history(history, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(history.history["loss"], label="train")
    ax1.plot(history.history["val_loss"], label="val")
    ax1.set_title(f"{title} - Loss"); ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
    ax1.legend(); ax1.grid(True)
    ax2.plot(history.history["accuracy"], label="train")
    ax2.plot(history.history["val_accuracy"], label="val")
    ax2.set_title(f"{title} - Accuracy"); ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy")
    ax2.legend(); ax2.grid(True)
    plt.tight_layout()
    plt.show()

plot_history(history_head, "Feature Extraction")

## 4) Phase 2 — Fine-Tune the Top Layers of ResNet50

Unfreeze the last third of the base, keep a very low learning rate, and continue training.

In [ ]:
# Unfreeze the top third of the base network
base.trainable = True
for layer in base.layers[: int(len(base.layers) * 0.67)]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR_FINE),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

cb_ft = [
    callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True, verbose=1),
    callbacks.ModelCheckpoint(os.path.join(MODELS_DIR, "resnet50_ft_best.keras"),
                              monitor="val_accuracy", save_best_only=True, verbose=1),
]

history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FT,
    callbacks=cb_ft,
    verbose=1,
)

In [ ]:
plot_history(history_ft, "Fine-Tuning")

## 5) Evaluate on the Test Set

In [ ]:
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

y_true = np.concatenate([np.argmax(y, axis=-1) for _, y in test_ds], axis=0)
y_prob = model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_prob, axis=-1)

report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3)
acc = accuracy_score(y_true, y_pred)
print(f"\nOverall Test Accuracy: {acc:.4f}\n")
print(report)

with open(os.path.join(RESULTS_DIR, "transfer_classification_report.txt"), "w", encoding="utf-8") as f:
    f.write(f"Test Accuracy: {acc:.4f}\n\n")
    f.write(report)

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(NUM_CLASSES), CLASS_NAMES, rotation=45, ha="right")
ax.set_yticks(range(NUM_CLASSES), CLASS_NAMES)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion Matrix")
fig.colorbar(im, ax=ax)
thresh = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, "%d" % cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "transfer_confusion_matrix.png"), dpi=120, bbox_inches="tight")
plt.show()

## 6) Save the Fine-Tuned Model

In [ ]:
model.save(os.path.join(MODELS_DIR, "cifar10_resnet50_finetuned.keras"))
print("Saved:", os.path.join(MODELS_DIR, "cifar10_resnet50_finetuned.keras"))

## 7) (Optional) Save Everything to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p "/content/drive/MyDrive/CIFAR10_Project"
!cp -r models results "/content/drive/MyDrive/CIFAR10_Project/"
print("Saved to /content/drive/MyDrive/CIFAR10_Project/")